# Fase 3 — Notebook 3: VAE — Entrenamiento y Generación de PNGs Sintéticos

**Objetivo:** Entrenar un Variational Autoencoder (VAE) convolucional con los
segmentos reales de **clase 1 (Con Ansiedad)** del conjunto de train, y generar
imágenes sintéticas suficientes para alcanzar ratio 1:1 en train.

**Inputs:**
- `Ansiedad/fase3/mini_cv/modelo_ganador.json` → filtro y duración del modelo ganador
- Split holdout JSON del notebook 1 → stems de train clase 1
- PNGs reales en `Clasificación Final/Ansiedad/specs/{filtro}/{dur}s/1/`

**Output:**
- PNGs sintéticos en `Ansiedad/fase3/sinteticos/{filtro}/{dur}s/1_sintetico/`

**Arquitectura VAE:**
```
Encoder: IMG → [Conv 32→64→128→256] → [FC→μ, FC→logσ²]  (dim_z=128)
Decoder: z   → [FC] → [ConvT 256→128→64→32→1] → IMG reconstituida
Loss   : BCE_reconstruccion + β·KL_divergencia  (β=1)
```

**Nota:** El VAE trabaja internamente en escala de grises y tamaño fijo
(IMG_H × VAE_W). Los PNGs generados se guardan a ese mismo tamaño.
CNN2D los maneja con AdaptiveAvgPool; AST los maneja con padding.

---
**Prerequisitos:** notebooks 1 y 2 completados.

In [30]:
# ─── Celda 1: Imports y configuración ─────────────────────────────────────
import re
import json
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ══════════════════════════════════════════════════════════════════
SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
SPLITS_DIR   = Path("Ansiedad/fase3/splits")
SINTETICOS_ROOT = Path("Ansiedad/fase3/sinteticos")
RESULTS_ROOT = Path("Ansiedad/fase3/vae")
MINI_CV_ROOT = Path("Ansiedad/fase3/mini_cv")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Hiperparametros VAE
DIM_Z      = 128      # dimension del espacio latente
BETA       = 1.0      # peso del termino KL (beta-VAE; 1.0 = VAE estandar)
VAE_EPOCHS = 200      # epocas de entrenamiento
VAE_BATCH  = 64
VAE_LR     = 1e-3
VAE_PATIENCE = 20     # early stopping sobre loss de reconstruccion

# El VAE trabaja en escala de grises y tamaño fijo
# VAE_W se calcula automaticamente como el ancho mas comun de los PNGs reales
# para no distorsionar la escala temporal al generar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# ── Cargar TODOS los candidatos del mini_cv ───────────────────────────────
todos_path = MINI_CV_ROOT / "todos_candidatos.json"
if todos_path.exists():
    with open(todos_path) as f:
        CANDIDATOS = json.load(f)["candidatos"]

print(f"\nCandidatos a procesar: {[c['id'] for c in CANDIDATOS]}")
print("="*60)

Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080

Candidatos a procesar: ['CNN2D_logmel_2s', 'AST_gammatone_2s', 'AST_gammatone_3s', 'AST_gammatone_4s']


In [31]:
# ─── Celda 2: Cargar PNGs reales de clase 1 del train ─────────────────────
import gc  # Para limpieza de memoria al final de cada candidato

def load_split_train_class1(filter_name, duration):
    """Devuelve solo los stems de clase 1 del conjunto de train."""
    json_path = SPLITS_DIR / f"split_{filter_name}_{duration}s.json"
    with open(json_path) as f:
        sp = json.load(f)
    stems  = [s for s, l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l == 1]
    stems0 = [s for s, l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l == 0]
    print(f"Train clase 1 (con ansiedad) : {len(stems)} pacientes")
    print(f"Train clase 0 (sin ansiedad) : {len(stems0)} pacientes")
    return stems, stems0, sp


def get_all_png_paths(stems, image_root, cls="1"):
    """Devuelve todos los paths PNG de los stems indicados."""
    img_map = {}
    d = image_root / cls
    for p in sorted(d.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem)
        img_map.setdefault(stem, []).append(p)
    paths = [p for stem in stems for p in img_map.get(stem, [])]
    return paths, img_map


def count_class0_segs(stems0, image_root):
    """Cuenta segmentos de clase 0 en train."""
    count = 0
    d = image_root / "0"
    img_map = {}
    for p in sorted(d.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem)
        img_map.setdefault(stem, []).append(p)
    for stem in stems0:
        count += len(img_map.get(stem, []))
    return count


for cand in CANDIDATOS:
    # Definimos las variables globales que usabas, ahora por candidato
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cand['id']}")
    print(f"  Filtro         : {FILTER_NAME}")
    print(f"  Duracion       : {DURATION}s")
    print(f"  IMG_H          : {IMG_H}px")
    print(f"{'='*60}")




    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"

    train_c1_stems, train_c0_stems, split_data = load_split_train_class1(
        FILTER_NAME, DURATION)

    real_paths_c1, img_map_c1 = get_all_png_paths(
        train_c1_stems, image_root, cls="1")

    n_segs_c0 = count_class0_segs(train_c0_stems, image_root)
    n_segs_c1 = len(real_paths_c1)
    n_sinteticos = max(0, n_segs_c0 - n_segs_c1)

    print(f"\nSegmentos reales clase 1 en train : {n_segs_c1:,}")
    print(f"Segmentos reales clase 0 en train : {n_segs_c0:,}")
    print(f"Sinteticos a generar (para 1:1)   : {n_sinteticos:,}")

    # Detectar ancho mas frecuente de los PNGs reales
    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]  # muestra de 50
    VAE_W  = int(np.median(widths))
    print(f"\nAncho mediano de PNGs reales : {VAE_W}px")
    print(f"Tamano interno del VAE       : {IMG_H}px x {VAE_W}px (H x W)")

    # Limpieza de memoria antes del siguiente candidato
    del train_c1_stems, train_c0_stems, real_paths_c1, img_map_c1, split_data
    gc.collect(); torch.cuda.empty_cache()


  CANDIDATO: CNN2D_logmel_2s
  Filtro         : log-mel
  Duracion       : 2s
  IMG_H          : 128px
Train clase 1 (con ansiedad) : 15 pacientes
Train clase 0 (sin ansiedad) : 48 pacientes

Segmentos reales clase 1 en train : 2,783
Segmentos reales clase 0 en train : 7,665
Sinteticos a generar (para 1:1)   : 4,882

Ancho mediano de PNGs reales : 63px
Tamano interno del VAE       : 128px x 63px (H x W)

  CANDIDATO: AST_gammatone_2s
  Filtro         : gammatone
  Duracion       : 2s
  IMG_H          : 128px
Train clase 1 (con ansiedad) : 15 pacientes
Train clase 0 (sin ansiedad) : 48 pacientes

Segmentos reales clase 1 en train : 2,783
Segmentos reales clase 0 en train : 7,665
Sinteticos a generar (para 1:1)   : 4,882

Ancho mediano de PNGs reales : 63px
Tamano interno del VAE       : 128px x 63px (H x W)

  CANDIDATO: AST_gammatone_3s
  Filtro         : gammatone
  Duracion       : 3s
  IMG_H          : 128px
Train clase 1 (con ansiedad) : 15 pacientes
Train clase 0 (sin ansiedad) :

In [32]:
# ─── Celda 3: Dataset para el VAE ─────────────────────────────────────────

# El VAE trabaja con imagenes en escala de grises, normalizadas a [0,1]
# Se hace resize interno al tamaño fijo del VAE para que el batch sea
# homogeneo. Los PNGs generados se guardan a este mismo tamaño.
class VAEDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert("RGB"))


for cand in CANDIDATOS:
    FILTER_NAME = cand["filter"]; DURATION = cand["dur"]; IMG_H = cand["img_h"]

    # Recalculamos VAE_W rápido para no depender de Celda 2 (seguro en Jupyter)
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    stems1 = [s for s,l in zip(json.load(open(SPLITS_DIR/f"split_{FILTER_NAME}_{DURATION}s.json"))["train"]["stems"],
                               json.load(open(SPLITS_DIR/f"split_{FILTER_NAME}_{DURATION}s.json"))["train"]["labels"]) if l==1]
    d1 = image_root / "1"; img_map_c1 = {}
    for p in sorted(d1.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c1.setdefault(stem, []).append(p)
    real_paths_c1 = [p for stem in stems1 for p in img_map_c1.get(stem, [])]
    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]
    VAE_W  = int(np.median(widths))


    VAE_TRANSFORM = T.Compose([
        T.Grayscale(num_output_channels=1),
        T.Resize((IMG_H, VAE_W)),
        T.ToTensor(),          # → (1, H, W) en [0, 1]
    ])

    vae_ds = VAEDataset(real_paths_c1, VAE_TRANSFORM)
    vae_dl = DataLoader(vae_ds, batch_size=VAE_BATCH, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=False)

    print(f"Dataset VAE: {len(vae_ds):,} imagenes de clase 1")
    print(f"Batches por epoch: {len(vae_dl)}")

    # Visualizar algunas muestras reales
    sample_batch = next(iter(vae_dl))
    fig, axes = plt.subplots(2, 5, figsize=(15, 5))
    fig.suptitle(f"Muestras reales clase 1 — {FILTER_NAME} {DURATION}s", fontsize=12)
    for i, ax in enumerate(axes.flat):
        if i < len(sample_batch):
            ax.imshow(sample_batch[i].squeeze().numpy(), cmap='inferno', aspect='auto')
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / "muestras_reales.png", dpi=100)
    plt.show(); plt.close()
    print("Muestras reales visualizadas")

    #Limpieza
    del vae_ds, vae_dl, real_paths_c1, sample_batch
    gc.collect(); torch.cuda.empty_cache()

Dataset VAE: 2,783 imagenes de clase 1
Batches por epoch: 44
Muestras reales visualizadas
Dataset VAE: 2,783 imagenes de clase 1
Batches por epoch: 44


/tmp/ipykernel_84809/1956130008.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


Muestras reales visualizadas
Dataset VAE: 1,847 imagenes de clase 1
Batches por epoch: 29
Muestras reales visualizadas
Dataset VAE: 1,381 imagenes de clase 1
Batches por epoch: 22
Muestras reales visualizadas


In [33]:
# ─── Celda 4: Arquitectura VAE ─────────────────────────────────────────────
#
# VAE convolucional para espectrogramas en escala de grises.
#
# Encoder: 1 canal → 4 bloques Conv (32→64→128→256, stride=2)
#          → flatten → FC → μ y log(σ²)
#
# Decoder: z → FC → reshape → 4 bloques ConvTranspose (256→128→64→32→1)
#          → sigmoid (salida en [0,1])
#
# El tamaño espacial tras el encoder es (H//16) x (W//16) gracias
# a 4 capas con stride=2. El decoder lo devuelve al tamaño original.

class Encoder(nn.Module):
    def __init__(self, in_h, in_w, dim_z):
        super().__init__()
        # 4 x stride-2 → reduce H y W a la 16ava parte
        self.conv = nn.Sequential(
            nn.Conv2d(1,   32,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(64),
            nn.Conv2d(64,  128, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(256),
        )
        self.h_enc = in_h // 16
        self.w_enc = in_w // 16
        flat_dim   = 256 * self.h_enc * self.w_enc
        self.fc_mu  = nn.Linear(flat_dim, dim_z)
        self.fc_lv  = nn.Linear(flat_dim, dim_z)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)


class Decoder(nn.Module):
    def __init__(self, in_h, in_w, dim_z):
        super().__init__()
        self.h_enc  = in_h // 16
        self.w_enc  = in_w // 16
        flat_dim    = 256 * self.h_enc * self.w_enc
        self.fc     = nn.Linear(dim_z, flat_dim)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(128),
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64,  32,  4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.BatchNorm2d(32),
            nn.ConvTranspose2d(32,  1,   4, stride=2, padding=1), nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 256, self.h_enc, self.w_enc)
        return self.deconv(h)


class VAE(nn.Module):
    def __init__(self, in_h, in_w, dim_z=DIM_Z):
        super().__init__()
        # Ajustar in_h e in_w al multiplo de 16 mas cercano hacia arriba
        self.in_h = ((in_h + 15) // 16) * 16
        self.in_w = ((in_w + 15) // 16) * 16
        self.dim_z   = dim_z
        self.encoder = Encoder(self.in_h, self.in_w, dim_z)
        self.decoder = Decoder(self.in_h, self.in_w, dim_z)

    def reparametrize(self, mu, logvar):
        """Truco de reparametrizacion: z = mu + eps*sigma."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        # Ajustar tamaño si no es multiplo de 16
        if x.shape[2] != self.in_h or x.shape[3] != self.in_w:
            x = F.interpolate(x, size=(self.in_h, self.in_w), mode='bilinear',
                              align_corners=False)
        mu, logvar = self.encoder(x)
        z          = self.reparametrize(mu, logvar)
        recon      = self.decoder(z)
        return recon, mu, logvar

    def sample(self, n, device):
        """Genera n muestras desde el prior N(0,I)."""
        z = torch.randn(n, self.dim_z).to(device)
        with torch.no_grad():
            return self.decoder(z)


def vae_loss(recon, x, mu, logvar, beta=BETA):
    """
    Loss = BCE_reconstruccion + beta * KL_divergencia
    KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    """
    # Alinear tamaño si hubo padding interno
    if recon.shape != x.shape:
        recon = F.interpolate(recon, size=x.shape[2:], mode='bilinear',
                              align_corners=False)
    bce = F.binary_cross_entropy(recon, x, reduction='sum') / x.size(0)
    kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return bce + beta * kl, bce, kl


for cand in CANDIDATOS:
    FILTER_NAME = cand["filter"]; DURATION = cand["dur"]; IMG_H = cand["img_h"]

    # Recalculamos VAE_W igual que en Celda 3 (consistencia)
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    stems1 = [s for s,l in zip(json.load(open(SPLITS_DIR/f"split_{FILTER_NAME}_{DURATION}s.json"))["train"]["stems"],
                               json.load(open(SPLITS_DIR/f"split_{FILTER_NAME}_{DURATION}s.json"))["train"]["labels"]) if l==1]
    d1 = image_root / "1"; img_map_c1 = {}
    for p in sorted(d1.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c1.setdefault(stem, []).append(p)
    real_paths_c1 = [p for stem in stems1 for p in img_map_c1.get(stem, [])]
    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]
    VAE_W  = int(np.median(widths))

    # Instanciar el VAE
    vae = VAE(in_h=IMG_H, in_w=VAE_W, dim_z=DIM_Z).to(DEVICE)
    n_params = sum(p.numel() for p in vae.parameters())
    print(f"VAE instanciado")
    print(f"  Tamaño interno : ({vae.in_h} x {vae.in_w})")
    print(f"  Dim latente    : {DIM_Z}")
    print(f"  Parametros     : {n_params:,}")

    del vae; gc.collect(); torch.cuda.empty_cache()

VAE instanciado
  Tamaño interno : (128 x 64)
  Dim latente    : 128
  Parametros     : 4,533,505
VAE instanciado
  Tamaño interno : (128 x 64)
  Dim latente    : 128
  Parametros     : 4,533,505
VAE instanciado
  Tamaño interno : (128 x 96)
  Dim latente    : 128
  Parametros     : 6,110,465
VAE instanciado
  Tamaño interno : (128 x 128)
  Dim latente    : 128
  Parametros     : 7,687,425


In [34]:
# ─── Celda 5: Entrenamiento del VAE ───────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

for cand in CANDIDATOS:
    # 1. Variables por candidato
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    
    # Recalcular VAE_W rápido para instanciar el VAE correcto
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    stems1 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==1]
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    d1 = image_root / "1"; img_map_c1 = {}
    for p in sorted(d1.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c1.setdefault(stem, []).append(p)
    real_paths_c1 = [p for stem in stems1 for p in img_map_c1.get(stem, [])]
    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]
    VAE_W  = int(np.median(widths)) if widths else 63

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cand['id']}")
    print(f"  VAE input: {IMG_H} x {VAE_W}")
    print(f"{'='*60}")
    
    # 2. Rutas únicas por candidato (evita sobrescritura)
    cand_vae_dir = RESULTS_ROOT / cand["id"]
    cand_vae_dir.mkdir(parents=True, exist_ok=True)
    vae_ckpt_path = cand_vae_dir / "vae_checkpoint.pt"
    
    # ✅ RECREAR Dataset y DataLoader por candidato (evita dimension mismatch)
    VAE_TRANSFORM = T.Compose([
        T.Grayscale(num_output_channels=1),
        T.Resize((IMG_H, VAE_W)),
        T.ToTensor(),
    ])

    class VAEDataset(Dataset):
        def __init__(self, paths, transform):
            self.paths, self.transform = paths, transform
        def __len__(self): return len(self.paths)
        def __getitem__(self, idx):
            return self.transform(Image.open(self.paths[idx]).convert("RGB"))

    vae_ds = VAEDataset(real_paths_c1, VAE_TRANSFORM)
    vae_dl = DataLoader(vae_ds, batch_size=VAE_BATCH, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=False)

    print(f"Dataset VAE: {len(vae_ds):,} imagenes de clase 1")
    print(f"Batches por epoch: {len(vae_dl)}")
    
    
    # 3️ Instanciar VAE (necesario porque la celda anterior pierde el scope al terminar el for)
    vae = VAE(in_h=IMG_H, in_w=VAE_W, dim_z=DIM_Z).to(DEVICE)
    n_params = sum(p.numel() for p in vae.parameters())
    print(f"VAE instanciado | Params: {n_params:,}")

    

    # Si ya hay un checkpoint entrenado, cargarlo directamente
    if vae_ckpt_path.exists():
        ckpt = torch.load(vae_ckpt_path, map_location=DEVICE)
        vae.load_state_dict(ckpt["model_state"])
        print(f"[CKPT] VAE cargado desde epoch {ckpt['epoch']} "
              f"(loss={ckpt['best_loss']:.4f})")
        train_vae = False
    else:
        train_vae = True
    
    if train_vae:
        optimizer = optim.Adam(vae.parameters(), lr=VAE_LR)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 'min', factor=0.5, patience=10, min_lr=1e-6)
    
        history = {"loss": [], "bce": [], "kl": []}
        best_loss, pat_cnt, best_state = float('inf'), 0, None
    
        print(f"Entrenando VAE — {VAE_EPOCHS} epocas max (patience={VAE_PATIENCE})")
        print(f"Beta={BETA}  DimZ={DIM_Z}  LR={VAE_LR}  Batch={VAE_BATCH}")
        print()
    
        t0 = time.time()
        for epoch in range(VAE_EPOCHS):
            vae.train()
            ep_loss, ep_bce, ep_kl = 0.0, 0.0, 0.0
    
            for xb in vae_dl:
                xb = xb.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                recon, mu, logvar = vae(xb)
                loss, bce, kl     = vae_loss(recon, xb, mu, logvar)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
                optimizer.step()
                ep_loss += loss.item()
                ep_bce  += bce.item()
                ep_kl   += kl.item()
    
            ep_loss /= len(vae_dl)
            ep_bce  /= len(vae_dl)
            ep_kl   /= len(vae_dl)
            scheduler.step(ep_loss)
    
            history["loss"].append(ep_loss)
            history["bce"].append(ep_bce)
            history["kl"].append(ep_kl)
    
            if ep_loss < best_loss:
                best_loss  = ep_loss
                pat_cnt    = 0
                best_state = {k: v.cpu().clone() for k, v in vae.state_dict().items()}
                mark = "v"
            else:
                pat_cnt += 1
                mark = f"({pat_cnt}/{VAE_PATIENCE})"
    
            if (epoch + 1) % 10 == 0 or pat_cnt >= VAE_PATIENCE:
                print(f"  Ep {epoch+1:4d}/{VAE_EPOCHS}  "
                      f"loss={ep_loss:.4f}  bce={ep_bce:.4f}  kl={ep_kl:.4f}  {mark}")
    
            if pat_cnt >= VAE_PATIENCE:
                print(f"  Early stopping en epoch {epoch+1}")
                break
    
        vae.load_state_dict(best_state)
        elapsed = time.time() - t0
        print(f"\nEntrenamiento completado: {elapsed/60:.1f} min")
        print(f"Mejor loss: {best_loss:.4f}")
    
        # Guardar checkpoint
        torch.save({
            "model_state": best_state,
            "epoch"      : epoch + 1,
            "best_loss"  : best_loss,
            "history"    : history,
            "config"     : {"in_h": IMG_H, "in_w": VAE_W, "dim_z": DIM_Z,
                            "filter": FILTER_NAME, "dur": DURATION},
        }, vae_ckpt_path)
        print(f"Checkpoint guardado: {vae_ckpt_path}")
    
        # Curva de loss
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(history["loss"], label="Loss total", color="navy")
        ax.plot(history["bce"],  label="BCE recon",  color="steelblue", linestyle="--")
        ax.plot(history["kl"],   label="KL",         color="tomato",    linestyle="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
        ax.set_title(f"Curva de entrenamiento VAE — {FILTER_NAME} {DURATION}s")
        plt.tight_layout()
        plt.savefig(RESULTS_ROOT / "vae_loss_curve.png", dpi=120)
        plt.show(); plt.close()
        
    # 4 Limpieza de memoria para el siguiente candidato
    del vae; gc.collect(); torch.cuda.empty_cache()
    print(f"  ✅ {cand['id']} completado. Memoria liberada.\n")


  CANDIDATO: CNN2D_logmel_2s
  VAE input: 128 x 63
Dataset VAE: 2,783 imagenes de clase 1
Batches por epoch: 44
VAE instanciado | Params: 4,533,505
Entrenando VAE — 200 epocas max (patience=20)
Beta=1.0  DimZ=128  LR=0.001  Batch=64

  Ep   10/200  loss=4721.7173  bce=4687.8277  kl=33.8895  v
  Ep   20/200  loss=4705.2519  bce=4672.0497  kl=33.2022  v
  Ep   30/200  loss=4697.2776  bce=4663.9700  kl=33.3075  v
  Ep   40/200  loss=4693.2155  bce=4658.8621  kl=34.3534  v
  Ep   50/200  loss=4689.5337  bce=4655.1421  kl=34.3916  (1/20)
  Ep   60/200  loss=4685.7050  bce=4651.4572  kl=34.2478  v
  Ep   70/200  loss=4683.5223  bce=4649.0772  kl=34.4451  (2/20)
  Ep   80/200  loss=4678.4958  bce=4643.9905  kl=34.5053  v
  Ep   90/200  loss=4680.4945  bce=4645.3746  kl=35.1199  (3/20)
  Ep  100/200  loss=4675.8337  bce=4640.5709  kl=35.2628  v
  Ep  110/200  loss=4675.8051  bce=4640.1622  kl=35.6429  (2/20)
  Ep  120/200  loss=4672.3581  bce=4636.2647  kl=36.0934  v
  Ep  130/200  loss=4671.

/tmp/ipykernel_84809/1053448914.py:152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


  Ep   10/200  loss=4950.7165  bce=4903.9091  kl=46.8074  v
  Ep   20/200  loss=4927.1207  bce=4883.8081  kl=43.3126  v
  Ep   30/200  loss=4918.9482  bce=4875.8715  kl=43.0767  v
  Ep   40/200  loss=4913.0709  bce=4870.7952  kl=42.2757  v
  Ep   50/200  loss=4909.3363  bce=4867.6651  kl=41.6711  (1/20)
  Ep   60/200  loss=4905.6314  bce=4863.3890  kl=42.2425  (2/20)
  Ep   70/200  loss=4902.8484  bce=4860.2175  kl=42.6310  (2/20)
  Ep   80/200  loss=4899.2618  bce=4856.3926  kl=42.8692  v
  Ep   90/200  loss=4898.3076  bce=4855.3124  kl=42.9952  (2/20)
  Ep  100/200  loss=4896.7739  bce=4853.5484  kl=43.2255  (1/20)
  Ep  110/200  loss=4894.3994  bce=4850.8369  kl=43.5625  (6/20)
  Ep  120/200  loss=4893.3390  bce=4849.7448  kl=43.5942  (6/20)
  Ep  130/200  loss=4890.0220  bce=4846.0862  kl=43.9358  v
  Ep  140/200  loss=4889.1122  bce=4845.1104  kl=44.0018  (5/20)
  Ep  150/200  loss=4889.4827  bce=4845.3091  kl=44.1736  (5/20)
  Ep  160/200  loss=4887.3249  bce=4843.1807  kl=44.144

In [35]:
# ─── Celda 6: Verificación visual — reconstrucción y muestras del prior ───

for cand in CANDIDATOS:
    # Variables por candidato
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    cand_vae_dir = RESULTS_ROOT / cand["id"]

    # Recalcular VAE_W y recargar dataloader/modelo (necesario si se ejecuta celda a celda)
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    stems1 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==1]
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    d1 = image_root / "1"; img_map_c1 = {}
    for p in sorted(d1.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c1.setdefault(stem, []).append(p)
    real_paths_c1 = [p for stem in stems1 for p in img_map_c1.get(stem, [])]
    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]
    VAE_W  = int(np.median(widths)) if widths else 63

    VAE_TRANSFORM = T.Compose([T.Grayscale(1), T.Resize((IMG_H, VAE_W)), T.ToTensor()])
    class VAEDataset(Dataset):
        def __init__(self, paths, transform): self.paths, self.transform = paths, transform
        def __len__(self): return len(self.paths)
        def __getitem__(self, idx): return self.transform(Image.open(self.paths[idx]).convert("RGB"))
    vae_dl = DataLoader(VAEDataset(real_paths_c1, VAE_TRANSFORM), batch_size=VAE_BATCH, shuffle=True, num_workers=4, pin_memory=True)

    ckpt_path = cand_vae_dir / "vae_checkpoint.pt"
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        vae = VAE(in_h=IMG_H, in_w=VAE_W, dim_z=DIM_Z).to(DEVICE)
        vae.load_state_dict(ckpt["model_state"])
    else:
        raise FileNotFoundError(f"Checkpoint no encontrado: {ckpt_path}. Corre primero la Celda 5.")

    print(f"\n{'='*60}")
    print(f"  VISUALIZACIÓN: {cand['id']}")
    print(f"{'='*60}")
    
    # CODIGO ORGINIAL:
    vae.eval()
    
    # a) Reconstrucciones de muestras reales
    with torch.no_grad():
        xb = next(iter(vae_dl))[:8].to(DEVICE)
        recon, _, _ = vae(xb)
        if recon.shape != xb.shape:
            recon = F.interpolate(recon, size=xb.shape[2:], mode='bilinear', align_corners=False)
    
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    fig.suptitle("Arriba: original  |  Abajo: reconstruccion VAE", fontsize=11)
    for i in range(8):
        axes[0, i].imshow(xb[i].cpu().squeeze().numpy(),    cmap='inferno', aspect='auto')
        axes[0, i].axis('off')
        axes[1, i].imshow(recon[i].cpu().squeeze().numpy(), cmap='inferno', aspect='auto')
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.savefig(cand_vae_dir / "vae_reconstrucciones.png", dpi=120)
    plt.show(); plt.close()
    
    # b) Muestras puras del prior
    with torch.no_grad():
        samples = vae.sample(10, DEVICE).cpu().squeeze().numpy()
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 5))
    fig.suptitle("Muestras sinteticas del VAE (prior N(0,I))", fontsize=11)
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i], cmap='inferno', aspect='auto')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(cand_vae_dir / "vae_muestras_sinteticas_preview.png", dpi=120)
    plt.show(); plt.close() 
    
    print("Verificacion visual completada")
    print("Revisar las imagenes antes de generar el dataset completo.")
    print("Si la calidad es muy baja, considera aumentar VAE_EPOCHS o DIM_Z.")
    
    # Limpieza
    del vae, vae_dl, real_paths_c1
    gc.collect(); torch.cuda.empty_cache()


  VISUALIZACIÓN: CNN2D_logmel_2s


/tmp/ipykernel_84809/2063871438.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


Verificacion visual completada
Revisar las imagenes antes de generar el dataset completo.
Si la calidad es muy baja, considera aumentar VAE_EPOCHS o DIM_Z.

  VISUALIZACIÓN: AST_gammatone_2s


/tmp/ipykernel_84809/2063871438.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


Verificacion visual completada
Revisar las imagenes antes de generar el dataset completo.
Si la calidad es muy baja, considera aumentar VAE_EPOCHS o DIM_Z.

  VISUALIZACIÓN: AST_gammatone_3s
Verificacion visual completada
Revisar las imagenes antes de generar el dataset completo.
Si la calidad es muy baja, considera aumentar VAE_EPOCHS o DIM_Z.

  VISUALIZACIÓN: AST_gammatone_4s
Verificacion visual completada
Revisar las imagenes antes de generar el dataset completo.
Si la calidad es muy baja, considera aumentar VAE_EPOCHS o DIM_Z.


In [36]:
# ─── Celda 7: Generación y guardado de PNGs sintéticos por candidato ───────
for cand in CANDIDATOS:
    cid         = cand["id"]
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cid}")
    print(f"  Filtro: {FILTER_NAME} | Duración: {DURATION}s")
    print(f"{'='*60}")

    # ── TU BLOQUE DE ~30 LÍNEAS PARA RECALCULAR VAE_W Y n_sinteticos ──────
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    stems1 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==1]
    stems0 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==0]

    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    d1 = image_root / "1"; img_map_c1 = {}
    for p in sorted(d1.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c1.setdefault(stem, []).append(p)
    real_paths_c1 = [p for stem in stems1 for p in img_map_c1.get(stem, [])]
    n_segs_c1 = len(real_paths_c1)

    d0 = image_root / "0"; img_map_c0 = {}; n_segs_c0 = 0
    for p in sorted(d0.glob("*.png")):
        stem = re.sub(r'_seg\d+$', '', p.stem); img_map_c0.setdefault(stem, []).append(p)
    for stem in stems0: n_segs_c0 += len(img_map_c0.get(stem, []))
    n_sinteticos = max(0, n_segs_c0 - n_segs_c1)

    widths = [Image.open(p).size[0] for p in real_paths_c1[:50]]
    VAE_W  = int(np.median(widths)) if widths else 63
    # ──────────────────────────────────────────────────────────────────────

    # Directorio y conteo (TUS PRINTS ORIGINALES INTACTOS)
    sintetico_dir = SINTETICOS_ROOT / FILTER_NAME / f"{DURATION}s" / "1_sintetico"
    sintetico_dir.mkdir(parents=True, exist_ok=True)
    existing = list(sintetico_dir.glob("*.png"))
    n_existentes = len(existing)
    n_faltantes  = max(0, n_sinteticos - n_existentes)

    print(f"Directorio sinteticos : {sintetico_dir}")
    print(f"Ya generados          : {n_existentes}")
    print(f"A generar             : {n_faltantes}")

    if n_faltantes == 0:
        print("\nYa se generaron todos los sinteticos necesarios.")
    else:
        # Cargar VAE específico de ESTE candidato
        vae_ckpt_path = RESULTS_ROOT / cid / "vae_checkpoint.pt"
        ckpt = torch.load(vae_ckpt_path, map_location=DEVICE)
        vae = VAE(in_h=IMG_H, in_w=VAE_W, dim_z=DIM_Z).to(DEVICE)
        vae.load_state_dict(ckpt["model_state"])
        vae.eval()

        GEN_BATCH = 128   # generar en lotes
        generados = 0
        img_idx   = n_existentes

        print(f"\nGenerando {n_faltantes} PNGs en lotes de {GEN_BATCH}...")
        t0 = time.time()

        while generados < n_faltantes:
            batch_n = min(GEN_BATCH, n_faltantes - generados)

            with torch.no_grad():
                imgs = vae.sample(batch_n, DEVICE)  # (B, 1, H, W) en [0,1]
                # Interpolar al tamaño original (IMG_H, VAE_W)
                imgs = F.interpolate(imgs, size=(IMG_H, VAE_W), mode='bilinear', align_corners=False)

            # Convertir a numpy y aplicar colormap inferno para RGB
            imgs_np = imgs.cpu().squeeze(1).numpy()  # (B, IMG_H, IMG_W)

            for i in range(batch_n):
                # Aplicar colormap inferno (igual que los espectrogramas reales)
                img_arr = imgs_np[i]                    # valores en [0,1]
                colored = plt.cm.inferno(img_arr)        # → (H, W, 4) RGBA en [0,1]
                colored_rgb = (colored[:, :, :3] * 255).astype(np.uint8)  # → RGB uint8

                img_pil = Image.fromarray(colored_rgb, mode='RGB')
                fname   = sintetico_dir / f"sintetico_{img_idx:06d}.png"
                img_pil.save(fname)
                img_idx += 1

            generados += batch_n
            if generados % (GEN_BATCH * 5) == 0 or generados >= n_faltantes:
                print(f"  Generados: {generados}/{n_faltantes}  ({time.time()-t0:.1f}s)")

        print(f"\nGeneracion completada en {(time.time()-t0)/60:.1f} min")
        
        # 🧹 Liberar memoria antes del siguiente candidato
        del vae; gc.collect(); torch.cuda.empty_cache()

    # Verificacion final (TUS PRINTS ORIGINALES INTACTOS)
    total_sinteticos = len(list(sintetico_dir.glob("*.png")))
    print(f"\nTotal PNGs en 1_sintetico/ : {total_sinteticos:,}")
    print(f"Clase 0 train segs         : {n_segs_c0:,}")
    print(f"Clase 1 train segs (real)  : {n_segs_c1:,}")
    print(f"Clase 1 train segs (total) : {n_segs_c1 + total_sinteticos:,}")
    ratio_real = n_segs_c0 / max(n_segs_c1, 1)
    ratio_aug  = n_segs_c0 / max(n_segs_c1 + total_sinteticos, 1)
    print(f"\nRatio C0/C1 antes augment  : {ratio_real:.2f}:1")
    print(f"Ratio C0/C1 despues augment: {ratio_aug:.2f}:1")


  CANDIDATO: CNN2D_logmel_2s
  Filtro: log-mel | Duración: 2s
Directorio sinteticos : Ansiedad/fase3/sinteticos/log-mel/2s/1_sintetico
Ya generados          : 0
A generar             : 4882

Generando 4882 PNGs en lotes de 128...
  Generados: 640/4882  (0.8s)
  Generados: 1280/4882  (1.5s)
  Generados: 1920/4882  (2.3s)
  Generados: 2560/4882  (3.0s)
  Generados: 3200/4882  (3.8s)
  Generados: 3840/4882  (4.5s)
  Generados: 4480/4882  (5.3s)
  Generados: 4882/4882  (5.8s)

Generacion completada en 0.1 min

Total PNGs en 1_sintetico/ : 4,882
Clase 0 train segs         : 7,665
Clase 1 train segs (real)  : 2,783
Clase 1 train segs (total) : 7,665

Ratio C0/C1 antes augment  : 2.75:1
Ratio C0/C1 despues augment: 1.00:1

  CANDIDATO: AST_gammatone_2s
  Filtro: gammatone | Duración: 2s
Directorio sinteticos : Ansiedad/fase3/sinteticos/gammatone/2s/1_sintetico
Ya generados          : 0
A generar             : 4882

Generando 4882 PNGs en lotes de 128...
  Generados: 640/4882  (0.7s)
  Genera

In [37]:
# ─── Celda 8: Resumen y verificacion visual de sinteticos guardados ────────
for cand in CANDIDATOS:
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    cand_dir    = RESULTS_ROOT / cand["id"]
    sintetico_dir = SINTETICOS_ROOT / FILTER_NAME / f"{DURATION}s" / "1_sintetico"

    # ── RECALCULAR CONTADORES (mismo bloque de ~15 líneas de la Celda 7) ──
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    stems1 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==1]
    stems0 = [s for s,l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l==0]
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    
    d1 = image_root / "1"; n_segs_c1 = 0
    for p in sorted(d1.glob("*.png")):
        if re.sub(r'_seg\d+$', '', p.stem) in stems1: n_segs_c1 += 1
        
    d0 = image_root / "0"; n_segs_c0 = 0
    for p in sorted(d0.glob("*.png")):
        if re.sub(r'_seg\d+$', '', p.stem) in stems0: n_segs_c0 += 1
        
    total_sinteticos = len(list(sintetico_dir.glob("*.png")))
    ratio_real = n_segs_c0 / max(n_segs_c1, 1)
    ratio_aug  = n_segs_c0 / max(n_segs_c1 + total_sinteticos, 1)
    # ──────────────────────────────────────────────────────────────────────

    # Mostrar algunos de los PNGs guardados para verificacion
    saved_paths = sorted(sintetico_dir.glob("*.png"))[:10]

    fig, axes = plt.subplots(2, 5, figsize=(15, 5))
    fig.suptitle(f"PNGs sinteticos guardados — {FILTER_NAME} {DURATION}s", fontsize=11)
    for i, ax in enumerate(axes.flat):
        if i < len(saved_paths):
            img = Image.open